In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [117]:
path = "../data/df_diabetic.csv"
df = pd.read_csv(path, low_memory=False)

with pd.option_context('display.max_columns', None):
    display(df.head())

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,0
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,0


In [119]:
def match_icd9(diag_col):
    #Matches all diagnoses to 
    diag_col = diag_col.str.replace(r'[E-V]\d*', '1000', regex = True).astype(float)

    conditions = [
        diag_col.between(1, 139),
        diag_col.between(140, 239),
        (diag_col.between(240, 279)) & (np.floor(diag_col) != 250),
        np.floor(diag_col) == 250,
        diag_col.between(280, 289),
        diag_col.between(290, 319),
        diag_col.between(320, 359),
        diag_col.between(360, 389),
        (diag_col.between(390, 459)) | (diag_col == 785),
        (diag_col.between(460, 519)) | (diag_col == 786),
        (diag_col.between(520, 579)) | (diag_col == 787),
        (diag_col.between(580, 629)) | (diag_col == 788),
        diag_col.between(630, 679),
        (diag_col.between(680, 709)) | (diag_col == 782),
        diag_col.between(710, 739),
        diag_col.between(740, 759),
        diag_col.between(800, 999),
        diag_col == 1000,
        diag_col.isna()
    ]

    labels = [
        'infectious and parasitic diseases',
        'neoplasms',
        'endocrine, nutritional, and metabolic diseases and immunity disorders',
        'diabetes mellitus',
        'diseases of the blood and blood-forming organs',
        'mental disorders',
        'diseases of the nervous system',
        'diseases of the sense organs',
        'diseases of the circulatory system',
        'diseases of the respiratory system',
        'diseases of the digestive system',
        'diseases of the genitourinary system',
        'complications of pregnancy, childbirth, and the puerperium',
        'diseases of the skin and subcutaneous tissue',
        'diseases of the musculoskeletal system and connective tissue',
        'congenital anomalies',
        'injury and poisoning',
        'external causes of injury and supplemental classification',
        'na'
    ]
    
    return pd.Series(np.select(conditions, labels, default = 'other'))

match_icd9(df['diag_1']).value_counts(normalize = True)

diseases of the circulatory system                                       0.299088
diseases of the respiratory system                                       0.141727
diseases of the digestive system                                         0.093106
diabetes mellitus                                                        0.086050
injury and poisoning                                                     0.068530
diseases of the genitourinary system                                     0.050282
diseases of the musculoskeletal system and connective tissue             0.048710
neoplasms                                                                0.033734
other                                                                    0.030747
infectious and parasitic diseases                                        0.027200
endocrine, nutritional, and metabolic diseases and immunity disorders    0.026551
diseases of the skin and subcutaneous tissue                             0.025686
mental disorders

In [111]:
def match_icd9_group(diag_col):
    diag_col = diag_col.str.replace(r'[E-V]\d*', '1000', regex = True).astype(float)

    conditions = [
        (diag_col.between(390, 459)) | (diag_col == 785),
        (diag_col.between(460, 519)) | (diag_col == 786),
        (diag_col.between(520, 579)) | (diag_col == 787),
        np.floor(diag_col) == 250,
        diag_col.between(800, 999),
        diag_col.between(710, 739),
        (diag_col.between(580, 629)) | (diag_col == 788),
        diag_col.between(140, 239),
        diag_col.isna()
    ]

    groups = [
        'circulatory', 
        'respiratory', 
        'digestive', 
        'diabetes', 
        'injury', 
        'musculoskeletal',
        'genitourinary',
        'neoplasms',
        'na'
    ]
    
    return pd.Series(np.select(conditions, groups, default = 'other'))

match_icd9_group(df['diag_2']).value_counts(normalize = True)

circulatory        0.313323
other              0.260868
diabetes           0.125716
respiratory        0.107072
genitourinary      0.082304
digestive          0.040985
neoplasms          0.025033
injury             0.023854
musculoskeletal    0.017337
nan                0.003509
Name: proportion, dtype: float64